# Advanced Decision Tree Classification

This notebook builds an **advanced Decision Tree classifier** using the Breast Cancer dataset from scikit-learn.

### Topics covered
- Data loading and exploration
- Train/test split
- Pipeline and feature scaling
- Hyperparameter tuning with `GridSearchCV`
- Cross-validation
- Feature importance
- Confusion matrix and classification report
- ROC-AUC and ROC curve
- Decision tree visualization
- Prediction on new samples
- Model saving with Joblib

> Note: Decision trees do not require feature scaling, but the pipeline demonstrates how preprocessing can be integrated safely into a machine-learning workflow.

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve
)
import joblib

print("Libraries imported successfully.")

In [ ]:
# Load dataset
data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

print("Dataset shape:", X.shape)
print("\nTarget classes:", dict(enumerate(data.target_names)))
print("\nFirst 5 rows:")
display(X.head())

In [ ]:
# Basic data inspection
print("Missing values:", X.isnull().sum().sum())
print("\nClass distribution:")
print(y.value_counts().rename(index=dict(enumerate(data.target_names))))

print("\nDataset statistics:")
display(X.describe().T.head(10))

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples :", X_test.shape[0])

In [ ]:
# Build an advanced pipeline
# Scaling is included to demonstrate a reusable preprocessing + model pipeline.
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", DecisionTreeClassifier(random_state=42))
])

pipeline.fit(X_train, y_train)

print("Initial Decision Tree trained successfully.")

In [ ]:
# Hyperparameter tuning using GridSearchCV
param_grid = {
    "classifier__criterion": ["gini", "entropy", "log_loss"],
    "classifier__max_depth": [3, 5, 7, 10, None],
    "classifier__min_samples_split": [2, 5, 10],
    "classifier__min_samples_leaf": [1, 2, 4],
    "classifier__max_features": [None, "sqrt", "log2"],
    "classifier__class_weight": [None, "balanced"]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="f1",
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print("Best parameters:")
print(grid_search.best_params_)
print("\nBest cross-validation F1 score:", round(grid_search.best_score_, 4))

In [ ]:
# Evaluate the best model
best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1 Score": f1_score(y_test, y_pred),
    "ROC-AUC": roc_auc_score(y_test, y_prob)
}

results = pd.DataFrame(
    {"Score": [round(v, 4) for v in metrics.values()]},
    index=metrics.keys()
)

display(results)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=data.target_names))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=data.target_names
)
disp.plot()
plt.title("Decision Tree - Confusion Matrix")
plt.show()

In [ ]:
# ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"Decision Tree (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Feature importance
tree_model = best_model.named_steps["classifier"]

importance = pd.Series(
    tree_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print("Top 15 important features:")
display(importance.head(15).to_frame("Importance"))

plt.figure(figsize=(10, 6))
importance.head(15).sort_values().plot(kind="barh")
plt.xlabel("Importance")
plt.title("Top 15 Decision Tree Feature Importances")
plt.tight_layout()
plt.show()

In [ ]:
# Visualize the trained decision tree
plt.figure(figsize=(22, 12))

plot_tree(
    tree_model,
    feature_names=X.columns,
    class_names=data.target_names,
    filled=True,
    rounded=True,
    max_depth=4,
    fontsize=8
)

plt.title("Decision Tree Structure (First 4 Levels)")
plt.show()

In [ ]:
# Cross-validation performance of the tuned model
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(
    best_model,
    X_train,
    y_train,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

print("Cross-validation accuracy scores:")
print(np.round(cv_scores, 4))
print("Mean CV accuracy:", round(cv_scores.mean(), 4))
print("Standard deviation:", round(cv_scores.std(), 4))

In [ ]:
# Predict new samples
# Here we use two existing test samples as realistic examples.
sample_data = X_test.iloc[:2]

predictions = best_model.predict(sample_data)
probabilities = best_model.predict_proba(sample_data)

prediction_table = pd.DataFrame({
    "Predicted Class": [data.target_names[p] for p in predictions],
    "Class 0 Probability": probabilities[:, 0],
    "Class 1 Probability": probabilities[:, 1]
})

display(prediction_table)

In [ ]:
# Save the complete trained pipeline
model_path = "advanced_decision_tree_model.joblib"
joblib.dump(best_model, model_path)

print(f"Model saved successfully as: {model_path}")

In [ ]:
# Load the saved model and verify prediction
loaded_model = joblib.load("advanced_decision_tree_model.joblib")

loaded_predictions = loaded_model.predict(X_test.iloc[:5])

print("Predictions from loaded model:")
print([data.target_names[p] for p in loaded_predictions])

## Conclusion

The notebook demonstrates an end-to-end advanced Decision Tree workflow:

1. Loaded and inspected a real classification dataset.
2. Split the data using stratification.
3. Built a reusable preprocessing/model pipeline.
4. Tuned multiple Decision Tree hyperparameters using `GridSearchCV`.
5. Evaluated the model using Accuracy, Precision, Recall, F1 and ROC-AUC.
6. Generated a confusion matrix and ROC curve.
7. Analyzed feature importance.
8. Visualized the decision tree.
9. Performed cross-validation.
10. Saved and reloaded the trained model.

### Important Decision Tree hyperparameters
- `max_depth`: Controls tree depth and helps prevent overfitting.
- `min_samples_split`: Minimum samples required to split a node.
- `min_samples_leaf`: Minimum samples required in a leaf.
- `criterion`: Measures split quality (`gini`, `entropy`, `log_loss`).
- `max_features`: Limits the number of features considered at each split.
- `class_weight`: Useful when classes are imbalanced.